In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from share.sumcheck import Verifier, Prover
from share.mlesumcheck import MLEProver, trans_tables
from share.interactive import Simulation, Party
from share.field import F
from share.poly import UniPoly
from share.mle import MLE
from share.matmul import MatmulProver, MatmulVerifier

from itertools import product
from random import randint, seed
from types import MethodType

In [7]:
A = [0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0]
A = [F(e) for e in A]


class CountTrianglesSimpleProver(MLEProver):

    def __init__(self, name, A):
        ln = self.ln = (len(A).bit_length() - 1) // 2
        Axy = [0] * 2**(3*ln)
        Ayz = [0] * 2**(3*ln)
        Axz = [0] * 2**(3*ln)

        for i in range(2**(3*ln)):
            x = i >> 2*ln
            y = (i >> ln) ^ (x << ln)
            z = i ^ (x << 2*ln) ^ (y << ln)
            Axy[i] = A[(x << ln) | y]
            Ayz[i] = A[(y << ln) | z]
            Axz[i] = A[(x << ln) | z]

        super().__init__(name, [Axy, Ayz, Axz])


class CountTrianglesSimpleVerifier(Verifier):
    def __init__(self, name, A, claim):
        ln = self.ln = (len(A).bit_length() - 1) // 2
        mle_A = MLE(A)
        g = lambda *x: mle_A(*x[:-ln])*mle_A(*x[ln:])*mle_A(*x[:ln],*x[-ln:])
        degs = [3] * (3*ln) # loose for check (true is 2)
        super().__init__(name, g, degs, claim)

prover = CountTrianglesSimpleProver("P", A)
verifier = CountTrianglesSimpleVerifier("V", A, prover.sum())

prover.set_dst(verifier)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()
print(prover.sum()/6)

{'P': None, 'V': True}
P -> V: ('g1', [F(220), F(212), F(116), F(2305843009213693883)])
V -> P: ('r1', F(2134139354768280835))
P -> V: ('g2', [F(2044451598876601400), F(1038389187893768281), F(2083808807884980787), F(569024440422851016)])
V -> P: ('r2', F(1636871794675947429))
P -> V: ('g3', [F(1078446097690992303), F(796202800441669121), F(1894945771481042852), F(2068832001595419545)])
V -> P: ('r3', F(1316587892191542527))
P -> V: ('g4', [F(1933627701500407872), F(862959281862259611), F(1075051398263821817), F(264061041491400539)])
V -> P: ('r4', F(1687634588898568434))
P -> V: ('g5', [F(1551598442714843867), F(1063509030399355981), F(1568402399628048994), F(760435541187228955)])
V -> P: ('r5', F(1787488633946274655))
P -> V: ('g6', [F(251305997022767679), F(406405299134822887), F(807024749600354589), F(1453164348419362785)])
V -> P: ('r6', F(767359944638474350))
P -> V: ('g7', [F(1278876933918485327), F(1141683133094266369), F(389383037061237678), F(1327819655033093205)])
V -> P: ('

In [8]:
class CountTrianglesEfficientProver(Party):

    def __init__(self, name, A):
        super().__init__(name)

        self.matmul = MatmulProver(name, A, A)
        self.mlesum = MLEProver(name, [self.matmul.C, A])

    def sum(self):
        return self.mlesum.sum()

    def set_dst(self, verifier):
        self.matmul.set_dst(verifier)
        self.mlesum.set_dst(verifier)

    def run(self):
        yield from self.mlesum.run()
        yield from self.matmul.run()


class CountTrianglesEfficientVerifier(Party):

    def __init__(self, name, A, claim):
        super().__init__(name)
        ln = self.ln = (len(A).bit_length() - 1) // 2
        self.A = A

        self.matmul = MatmulVerifier(name, (), A, A)
        self.mlesum = Verifier(name, lambda _: _, [2, 2] * ln, claim)

    def set_dst(self, prover):
        self.matmul.set_dst(prover)

    def run(self):
        out = yield from self.mlesum.loop()
        if out is None:
            return False
        rs, last = out
        
        ln = self.ln
        self.matmul.r = (rs[:ln], rs[ln:])
        result = yield from self.matmul.run()
        return (self.matmul.sum * MLE(self.A)(*rs) == last) and result


prover = CountTrianglesEfficientProver("P", A)
verifier = CountTrianglesEfficientVerifier("V", A, prover.sum())

prover.set_dst(verifier)
verifier.set_dst(prover)

seed(1337)
sim = Simulation([prover, verifier])
sim.run()
print(sim.result)
sim.print_log()
sim.print_times()
print(prover.sum()/6)

{'P': None, 'V': True}
P -> V: ('g1', [F(220), F(212), F(116)])
V -> P: ('r1', F(2134139354768280835))
P -> V: ('g2', [F(2044451598876601400), F(1038389187893768281), F(2083808807884980787)])
V -> P: ('r2', F(1636871794675947429))
P -> V: ('g3', [F(1078446097690992303), F(796202800441669121), F(1894945771481042852)])
V -> P: ('r3', F(1316587892191542527))
P -> V: ('g4', [F(1933627701500407872), F(862959281862259611), F(1075051398263821817)])
V -> P: ('r4', F(1687634588898568434))
P -> V: ('g5', [F(1551598442714843867), F(1063509030399355981), F(1568402399628048994)])
V -> P: ('r5', F(1787488633946274655))
P -> V: ('g6', [F(251305997022767679), F(406405299134822887), F(807024749600354589)])
V -> P: ('r6', F(767359944638474350))
P -> V: ('g7', [F(1278876933918485327), F(1141683133094266369), F(389383037061237678)])
V -> P: ('r7', F(1463919589942000245))
P -> V: ('g8', [F(713950542394105989), F(1904202692828832813), F(1701414586910406313)])
V -> P: ([F(2134139354768280835), F(163687179467